# Modelo Bayesian: AODE

Notebook elaborado siguiendo la rúbrica de la asignatura (ver `../../rubrica.md`).

**Autor:** Grupo E
**Fecha:** 13/06/2026

## 1. Descripción

**AODE** (*Averaged One-Dependence Estimators*) es un clasificador bayesiano supervisado que extiende la idea de los modelos de dependencia de una sola variable. Su objetivo es estimar la clase más probable de una instancia promediando varios estimadores de dependencia condicional.

Dada una instancia con atributos discretos $X = (x_1, x_2, \dots, x_m)$, AODE calcula la probabilidad posterior de cada clase $C$ mediante una combinación de modelos donde cada atributo puede actuar como un **superparent**. El resultado final es una distribución posterior sobre las clases y una predicción basada en la clase con mayor probabilidad.

Este enfoque busca capturar dependencias locales entre atributos sin renunciar a la simplicidad del marco bayesiano.

### Casos de uso típicos
- Clasificación de correos o textos discretizados.
- Diagnóstico médico con variables categóricas.
- Detección de fraude con atributos discretos.
- Clasificación de clientes o riesgo crediticio.
- Análisis de patrones en dominios tabulares.

## 2. Bibtex y Referencias

### BibTeX
```bibtex
@article{webb2005aode,
  title   = {Not So Naive Bayes: Aggregating One-Dependence Estimators},
  author  = {Webb, Geoffrey I. and Boughton, Janice R. and Wang, Zhihua},
  journal = {Machine Learning},
  volume  = {58},
  number  = {1},
  pages   = {5--24},
  year    = {2005},
  doi     = {10.1007/s10994-005-4258-6}
}

@article{webb2012ande,
  title   = {Learning by Extrapolation from Marginal to Full-Multivariate Probability Distributions: Decreasingly Naive Bayesian Classification},
  author  = {Webb, Geoffrey I. and Boughton, Janice R. and Zheng, Fei and Ting, Kai Ming and Salem, Houssam},
  journal = {Machine Learning},
  volume  = {86},
  number  = {2},
  pages   = {233--272},
  year    = {2012},
  doi     = {10.1007/s10994-011-5263-6}
}
```

### APA
- Webb, G. I., Boughton, J. R., & Wang, Z. (2005). *Not so naive Bayes: Aggregating one-dependence estimators*. **Machine Learning, 58**(1), 5–24. https://doi.org/10.1007/s10994-005-4258-6
- Webb, G. I., Boughton, J. R., Zheng, F., Ting, K. M., & Salem, H. (2012). *Learning by extrapolation from marginal to full-multivariate probability distributions: Decreasingly naive Bayesian classification*. **Machine Learning, 86**(2), 233–272. https://doi.org/10.1007/s10994-011-5263-6

## 3. Tipo de Modelo

| Criterio | Clasificación |
|---|---|
| **Método de aprendizaje** | Supervisado |
| **Por parámetros** | Paramétrico |
| **Datos de aprendizaje** | Offline (batch) |
| **Resultado del entrenamiento** | Clasificador bayesiano probabilístico |

Notas:
- **Supervisado** porque aprende a partir de atributos y etiquetas de clase.
- **Paramétrico** porque estima un número fijo de probabilidades condicionales a partir de los datos, igual que Naive Bayes.
- **Offline** porque necesita estimar frecuencias a partir del conjunto de entrenamiento.
- El resultado es una función de decisión probabilística basada en clases posteriores.

## 4. Algoritmo de Entrenamiento

AODE construye varios estimadores de dependencia de una sola variable y luego promedia sus salidas para obtener una predicción robusta.

### Pseudocódigo
```text
Entrada: D = {(x^(i), c^(i))}, atributo clase C, umbral m
Salida: clasificador AODE

1. Contar frecuencias de clases y atributos discretos.
2. Para una instancia nueva x:
   a. Identificar los atributos candidatos a superparent.
   b. Para cada superparent X_j que cumpla el umbral m:
      - Calcular P(C)
      - Calcular P(X_j | C)
      - Calcular P(X_k | X_j, C) para cada atributo restante
      - Obtener un estimador de dependencia de una sola variable
   c. Promediar todos los estimadores válidos.
3. Elegir la clase con mayor probabilidad posterior.
```

### Componentes principales
- **Priors de clase:** frecuencia relativa de cada clase.
- **Probabilidades condicionales:** estimaciones de atributos dado clase o dado superparent y clase.
- **Superparent:** atributo que condiciona a los demás atributos en cada estimador.
- **Promedio de estimadores:** estabiliza la predicción y reduce la varianza.

### Fórmula general

Para una instancia $x = (x_1, x_2, \dots, x_m)$, AODE estima:

$$
P(C \mid x) \propto \sum_{j \in S} P(C) P(x_j \mid C) \prod_{i \neq j} P(x_i \mid x_j, C)
$$

donde $S$ es el conjunto de atributos válidos como superparents.

### Métricas clave

- **Probabilidad posterior:** $P(C \mid x)$
- **Exactitud:** proporción de predicciones correctas
- **Precisión:** proporción de predicciones positivas correctas
- **Recall:** proporción de positivos reales detectados
- **F1-score:** media armónica entre precisión y recall

## 5. Supuestos y Restricciones

- **Atributos discretos o categóricos:** las variables continuas deben discretizarse previamente.
- **Etiquetas de clase disponibles:** el modelo requiere datos etiquetados para aprender.
- **Suficiente soporte por atributo:** si un atributo tiene pocas observaciones, su contribución puede ser inestable.
- **Independencia condicional aproximada:** el enfoque mejora la flexibilidad, pero sigue dependiendo de estimaciones bayesianas.
- **Sensibilidad al tamaño de muestra:** con pocos datos, las frecuencias pueden ser ruidosas.
- **Necesidad de suavizado:** Laplace u otro suavizado evita probabilidades nulas.
- **Variables faltantes:** conviene imputarlas o tratarlas explícitamente antes del entrenamiento.

## 6. Tests / Métricas de validación

AODE se valida como clasificador mediante métricas de desempeño predictivo:

- **Exactitud (accuracy)** — porcentaje total de aciertos.
- **Precisión (precision)** — calidad de las predicciones positivas.
- **Recall** — capacidad para recuperar la clase positiva.
- **F1-score** — balance entre precisión y recall.
- **Matriz de confusión** — distribución de aciertos y errores por clase.
- **Probabilidades posteriores** — útil para interpretar la confianza de la predicción.

En problemas multiclase también pueden usarse métricas macro y micro promediadas.

---
## 7. Implementación práctica

Usaremos un conjunto de datos categórico pequeño para ilustrar el algoritmo AODE desde cero.

### 7.1 Instalación e imports

In [1]:
import pandas as pd
import numpy as np

print('Librerías cargadas correctamente')

Librerías cargadas correctamente


### 7.2 Dataset de ejemplo

Usamos un dataset clásico de clima y decisión, con atributos discretos y una variable objetivo binaria.

In [2]:
datos = pd.DataFrame([
    ['soleado', 'caliente', 'alta', 'falso', 'no'],
    ['soleado', 'caliente', 'alta', 'verdadero', 'no'],
    ['nublado', 'caliente', 'alta', 'falso', 'si'],
    ['lluvioso', 'templado', 'alta', 'falso', 'si'],
    ['lluvioso', 'frio', 'normal', 'falso', 'si'],
    ['lluvioso', 'frio', 'normal', 'verdadero', 'no'],
    ['nublado', 'frio', 'normal', 'verdadero', 'si'],
    ['soleado', 'templado', 'alta', 'falso', 'no'],
    ['soleado', 'frio', 'normal', 'falso', 'si'],
    ['lluvioso', 'templado', 'normal', 'falso', 'si'],
    ['soleado', 'templado', 'normal', 'verdadero', 'si'],
    ['nublado', 'templado', 'alta', 'verdadero', 'si'],
    ['nublado', 'caliente', 'normal', 'falso', 'si'],
    ['lluvioso', 'templado', 'alta', 'verdadero', 'no'],
], columns=['cielo', 'temperatura', 'humedad', 'viento', 'jugar'])

datos

,cielo,temperatura,humedad,viento,jugar
0,soleado,caliente,alta,falso,no
1,soleado,caliente,alta,verdadero,no
2,nublado,caliente,alta,falso,si
3,lluvioso,templado,alta,falso,si
4,lluvioso,frio,normal,falso,si
5,lluvioso,frio,normal,verdadero,no
6,nublado,frio,normal,verdadero,si
7,soleado,templado,alta,falso,no
8,soleado,frio,normal,falso,si
9,lluvioso,templado,normal,falso,si


### 7.3 División entrenamiento y prueba

Separaremos una parte de los datos para evaluar el modelo.

In [3]:
def dividir_datos(df, proporcion_prueba=0.3, semilla=42):
    rng = np.random.default_rng(semilla)
    indices = rng.permutation(len(df))
    n_prueba = max(1, int(len(df) * proporcion_prueba))
    idx_prueba = indices[:n_prueba]
    idx_entrenamiento = indices[n_prueba:]
    return df.iloc[idx_entrenamiento].reset_index(drop=True), df.iloc[idx_prueba].reset_index(drop=True)

train_df, test_df = dividir_datos(datos, proporcion_prueba=0.3, semilla=42)

print(f'Tamaño entrenamiento: {len(train_df)}')
print(f'Tamaño prueba: {len(test_df)}')
train_df

Tamaño entrenamiento: 10
Tamaño prueba: 4


,cielo,temperatura,humedad,viento,jugar
0,soleado,caliente,alta,falso,no
1,nublado,templado,alta,verdadero,si
2,lluvioso,templado,alta,verdadero,no
3,soleado,templado,normal,verdadero,si
4,lluvioso,frio,normal,verdadero,no
5,nublado,caliente,alta,falso,si
6,lluvioso,frio,normal,falso,si
7,nublado,caliente,normal,falso,si
8,soleado,caliente,alta,verdadero,no
9,soleado,frio,normal,falso,si


### 7.4 Implementación de AODE

In [4]:
from collections import defaultdict

class AODEClassifier:
    def __init__(self, min_superparent_count=1, alpha=1.0):
        self.min_superparent_count = min_superparent_count
        self.alpha = alpha
        self.target_col = None
        self.feature_cols = None
        self.class_values = None
        self.feature_values = None
        self.train_df = None

    def fit(self, df, target_col):
        self.train_df = df.copy()
        self.target_col = target_col
        self.feature_cols = [col for col in df.columns if col != target_col]
        self.class_values = sorted(df[target_col].unique())
        self.feature_values = {
            col: sorted(df[col].dropna().unique())
            for col in self.feature_cols
        }
        return self

    def _laplace(self, count, total, cardinality):
        return (count + self.alpha) / (total + self.alpha * cardinality)

    def _count(self, conditions):
        mask = pd.Series(True, index=self.train_df.index)
        for col, value in conditions.items():
            mask &= self.train_df[col].eq(value)
        return int(mask.sum())

    def _posterior_for_class(self, row, class_value):
        class_count = self._count({self.target_col: class_value})
        total_count = len(self.train_df)
        if class_count == 0:
            return 0.0

        estimators = []
        for superparent in self.feature_cols:
            superparent_value = row[superparent]
            superparent_count = self._count({self.target_col: class_value, superparent: superparent_value})

            if superparent_count < self.min_superparent_count:
                continue

            score = self._laplace(class_count, total_count, len(self.class_values))
            score *= self._laplace(superparent_count, class_count, len(self.feature_values[superparent]))

            for feature in self.feature_cols:
                if feature == superparent:
                    continue
                feature_value = row[feature]
                joint_count = self._count({
                    self.target_col: class_value,
                    superparent: superparent_value,
                    feature: feature_value,
                })
                score *= self._laplace(joint_count, superparent_count, len(self.feature_values[feature]))

            estimators.append(score)

        if not estimators:
            score = self._laplace(class_count, total_count, len(self.class_values))
            for feature in self.feature_cols:
                feature_value = row[feature]
                feature_count = self._count({self.target_col: class_value, feature: feature_value})
                score *= self._laplace(feature_count, class_count, len(self.feature_values[feature]))
            return score

        return sum(estimators) / len(estimators)

    def predict_proba_one(self, row):
        raw_scores = {}
        for class_value in self.class_values:
            raw_scores[class_value] = self._posterior_for_class(row, class_value)

        total = sum(raw_scores.values())
        if total == 0:
            uniform = 1 / len(self.class_values)
            return {class_value: uniform for class_value in self.class_values}

        return {class_value: score / total for class_value, score in raw_scores.items()}

    def predict_one(self, row):
        proba = self.predict_proba_one(row)
        return max(proba, key=proba.get)

    def predict(self, df):
        return df.apply(self.predict_one, axis=1)

modelo = AODEClassifier(min_superparent_count=1, alpha=1.0)
modelo.fit(train_df, target_col='jugar')

print('Modelo AODE entrenado correctamente')

Modelo AODE entrenado correctamente


### 7.5 Predicción sobre el conjunto de prueba

In [5]:
resultado = test_df.copy()
resultado['prediccion'] = modelo.predict(test_df)
resultado['acierto'] = resultado['jugar'] == resultado['prediccion']
resultado

,cielo,temperatura,humedad,viento,jugar,prediccion,acierto
0,nublado,frio,normal,verdadero,si,no,False
1,soleado,templado,alta,falso,no,no,True
2,lluvioso,templado,normal,falso,si,si,True
3,lluvioso,templado,alta,falso,si,no,False


### 7.6 Métricas de evaluación

In [6]:
def matriz_confusion_binaria(y_real, y_pred, clase_positiva='si'):
    tp = int(((y_real == clase_positiva) & (y_pred == clase_positiva)).sum())
    tn = int(((y_real != clase_positiva) & (y_pred != clase_positiva)).sum())
    fp = int(((y_real != clase_positiva) & (y_pred == clase_positiva)).sum())
    fn = int(((y_real == clase_positiva) & (y_pred != clase_positiva)).sum())
    return tp, tn, fp, fn


def metricas_binarias(y_real, y_pred, clase_positiva='si'):
    tp, tn, fp, fn = matriz_confusion_binaria(y_real, y_pred, clase_positiva)
    total = tp + tn + fp + fn
    accuracy = (tp + tn) / total if total else 0
    precision = tp / (tp + fp) if (tp + fp) else 0
    recall = tp / (tp + fn) if (tp + fn) else 0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) else 0
    return {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'tp': tp,
        'tn': tn,
        'fp': fp,
        'fn': fn,
    }

metricas = metricas_binarias(resultado['jugar'], resultado['prediccion'], clase_positiva='si')

print(f"Exactitud: {metricas['accuracy']:.3f}")
print(f"Precisión: {metricas['precision']:.3f}")
print(f"Recall: {metricas['recall']:.3f}")
print(f"F1-score: {metricas['f1']:.3f}")

cm = pd.DataFrame(
    [[metricas['tn'], metricas['fp']], [metricas['fn'], metricas['tp']]],
    index=['Real: no', 'Real: si'],
    columns=['Pred: no', 'Pred: si']
)

cm

Exactitud: 0.500
Precisión: 1.000
Recall: 0.333
F1-score: 0.500


,Pred: no,Pred: si
Real: no,1,0
Real: si,2,1


### 7.7 Resultados del modelo

Mostramos la predicción y la probabilidad posterior para cada clase en cada instancia de prueba.

In [7]:
for idx, fila in test_df.iterrows():
    proba = modelo.predict_proba_one(fila)
    pred = max(proba, key=proba.get)
    print(f"Instancia {idx + 1} -> clase real: {fila['jugar']} | predicción: {pred} | probabilidades: {proba}")

Instancia 1 -> clase real: si | predicción: no | probabilidades: {'no': 0.5218804410911201, 'si': 0.4781195589088799}
Instancia 2 -> clase real: no | predicción: no | probabilidades: {'no': 0.6017635843660629, 'si': 0.3982364156339371}
Instancia 3 -> clase real: si | predicción: si | probabilidades: {'no': 0.3306914800230744, 'si': 0.6693085199769255}
Instancia 4 -> clase real: si | predicción: no | probabilidades: {'no': 0.6098646228461461, 'si': 0.3901353771538538}


### 7.8 Traducción de resultados

Interpretación en español de los resultados obtenidos:

- El modelo AODE construyó estimaciones posteriores para cada clase usando atributos discretos.
- La clase con mayor probabilidad posterior fue la seleccionada como predicción final.
- La matriz de confusión permite ver cuántos casos fueron clasificados correctamente y cuántos se confundieron.
- Si la exactitud es alta, el clasificador está capturando bien la estructura del conjunto de datos.
- Las probabilidades posteriores son útiles para entender el grado de confianza de cada decisión.

## 8. Conclusión

- AODE es un clasificador bayesiano **supervisado** que combina varios estimadores de dependencia de una sola variable.
- Su ventaja principal es que modela dependencias entre atributos sin abandonar la simplicidad probabilística.
- El uso de suavizado evita probabilidades nulas y mejora la robustez del modelo.
- AODE funciona mejor con atributos discretos y suficiente cantidad de datos etiquetados.
- En aplicaciones tabulares, ofrece una alternativa interpretable para clasificación probabilística.